# Personal Finance & Spending Analysis
## 03 — Exploratory Data Analysis (EDA)

**Input:** `data/cleaned_data.csv`  
**Outputs:** `reports/eda_summary.csv`, `reports/business_insights.csv`, charts in `visualizations/`

### Business questions
1. What does income distribution look like?
2. Which expense categories account for the largest spending?
3. How does income relate to disposable income and desired savings?
4. Does age relate to spending and savings patterns?
5. How do income and expenses differ by occupation?
6. How do financial patterns differ across city tiers?
7. Which categories show the greatest potential savings?
8. How closely does disposable income align with an expense-based estimate?

Use descriptive comparisons, distinguish observed data from derived estimates, and avoid causal claims. Run Notebook 02 first, then run all cells here from top to bottom.

## 1. Import Libraries and Configure Paths

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (10, 5)

In [ ]:
from pathlib import Path
HERE = Path.cwd()
PROJECT_ROOT = HERE if (HERE / "data").exists() else HERE.parent
print("Project root:", PROJECT_ROOT.resolve())

DATA_PATH = PROJECT_ROOT / "data" / "cleaned_data.csv"
REPORTS_DIR = PROJECT_ROOT / "reports"
CHARTS_DIR = PROJECT_ROOT / "visualizations"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
CHARTS_DIR.mkdir(parents=True, exist_ok=True)

## 2. Load and Validate Cleaned Dataset

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Cleaned data not found: {DATA_PATH.resolve()}. Run 02_data_cleaning.ipynb first."
    )
df = pd.read_csv(DATA_PATH)
print(f"Dataset size: {len(df):,} rows × {df.shape[1]:,} columns")
display(df.head())
df.info()

In [ ]:
required_columns = ["income"]
missing_required = [c for c in required_columns if c not in df.columns]
if missing_required:
    raise ValueError(f"Required columns missing: {missing_required}")
print("Required analysis fields are available.")

## 3. Define Analysis Variables and Derived Metrics

In [ ]:
expense_columns = [
    "rent", "loan_repayment", "insurance", "groceries", "transport",
    "eating_out", "entertainment", "utilities", "healthcare",
    "education", "miscellaneous"
]
expense_columns = [c for c in expense_columns if c in df.columns]
potential_savings_columns = [
    c for c in df.columns if c.startswith("potential_savings_")
]
categorical_columns = df.select_dtypes(
    include=["object", "category", "string", "bool"]
).columns.tolist()

print("Expense columns:", expense_columns)
print("Potential savings columns:", potential_savings_columns)
print("Categorical columns:", categorical_columns)

In [ ]:
# Count reported expense categories for each row.
if expense_columns:
    df["analysis_expense_categories_reported"] = df[expense_columns].notna().sum(axis=1)
    # Only calculate a complete total when every available expense category is present.
    df["analysis_total_expenses"] = df[expense_columns].sum(
        axis=1, min_count=len(expense_columns)
    )
    df["analysis_expense_data_complete"] = (
        df["analysis_expense_categories_reported"] == len(expense_columns)
    )
else:
    df["analysis_expense_categories_reported"] = 0
    df["analysis_total_expenses"] = np.nan
    df["analysis_expense_data_complete"] = False

df["analysis_expense_ratio_pct"] = (
    df["analysis_total_expenses"] / df["income"].replace(0, np.nan) * 100
)
df["analysis_estimated_remaining_income"] = (
    df["income"] - df["analysis_total_expenses"]
)
print("Analysis-only metrics created; source fields were not overwritten.")
display(df[[
    "income", "analysis_expense_categories_reported",
    "analysis_total_expenses", "analysis_estimated_remaining_income"
]].head())

## 4. Dataset Overview and Data Quality Snapshot

In [ ]:
overview = pd.DataFrame({
    "metric": [
        "Records", "Columns including analysis metrics", "Missing cells",
        "Duplicate rows", "Expense categories available",
        "Potential-savings categories available", "Complete expense records"
    ],
    "value": [
        len(df), df.shape[1], int(df.isna().sum().sum()),
        int(df.duplicated().sum()), len(expense_columns),
        len(potential_savings_columns),
        int(df["analysis_expense_data_complete"].sum())
    ]
})
display(overview)

## 5. Business Question 1 — Income Distribution

In [ ]:
display(df["income"].describe().to_frame("income").T)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df["income"].dropna(), bins=30, kde=True, ax=axes[0])
axes[0].set_title("Income Distribution")
sns.boxplot(x=df["income"], ax=axes[1])
axes[1].set_title("Income — Outlier Review")
plt.tight_layout()
plt.savefig(CHARTS_DIR / "01_income_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

Interpretation: Compare mean and median. A higher mean may indicate right skew. Outliers require investigation, not automatic deletion.

## 6. Business Question 2 — Expense Categories

In [ ]:
if not expense_columns:
    print("No recognized expense columns found.")
else:
    expense_summary = pd.DataFrame({
        "reported_record_count": df[expense_columns].count(),
        "total_reported_spending": df[expense_columns].sum(min_count=1),
        "average_spending_when_reported": df[expense_columns].mean(),
        "median_spending_when_reported": df[expense_columns].median()
    }).sort_values("total_reported_spending", ascending=False)

    grand_total = expense_summary["total_reported_spending"].sum()
    expense_summary["share_of_reported_spending_pct"] = (
        expense_summary["total_reported_spending"] / grand_total * 100
        if pd.notna(grand_total) and grand_total != 0 else np.nan
    )
    display(expense_summary)

    ax = expense_summary["total_reported_spending"].sort_values().plot(
        kind="barh", figsize=(10, 6), title="Total Reported Spending by Category"
    )
    ax.set_xlabel("Total reported spending")
    plt.tight_layout()
    plt.savefig(CHARTS_DIR / "02_expense_category_totals.png", dpi=150, bbox_inches="tight")
    plt.show()

Interpretation: Category totals use available observations. Different missing-value counts can affect comparisons, so inspect `reported_record_count` alongside totals.

## 7. Business Question 3 — Income, Disposable Income, and Savings

In [ ]:
relationship_columns = [
    c for c in [
        "income", "disposable_income", "desired_savings",
        "desired_savings_percentage", "analysis_total_expenses"
    ] if c in df.columns
]
display(df[relationship_columns].corr(numeric_only=True))

if "disposable_income" in df.columns:
    plot_data = df[["income", "disposable_income"]].dropna()
    if not plot_data.empty:
        fig, ax = plt.subplots(figsize=(8, 6))
        sns.regplot(data=plot_data, x="income", y="disposable_income",
                    scatter_kws={"alpha": 0.35},
                    line_kws={"color": "red"}, ax=ax)
        ax.set_title("Income vs. Disposable Income")
        plt.tight_layout()
        plt.savefig(CHARTS_DIR / "03_income_vs_disposable_income.png", dpi=150, bbox_inches="tight")
        plt.show()

if "desired_savings" in df.columns:
    plot_data = df[["income", "desired_savings"]].dropna()
    if not plot_data.empty:
        fig, ax = plt.subplots(figsize=(8, 6))
        sns.scatterplot(data=plot_data, x="income", y="desired_savings", alpha=0.45, ax=ax)
        ax.set_title("Income vs. Desired Savings")
        plt.tight_layout()
        plt.savefig(CHARTS_DIR / "04_income_vs_desired_savings.png", dpi=150, bbox_inches="tight")
        plt.show()

Interpretation: Examine correlation direction and strength, plus scatterplot spread. Correlation does not prove causation.

## 8. Business Question 4 — Age Groups

In [ ]:
if "age" not in df.columns:
    print("Age column unavailable; skipped.")
else:
    age_bins = [0, 24, 34, 44, 54, 64, 120]
    age_labels = ["0–24", "25–34", "35–44", "45–54", "55–64", "65+"]
    df["analysis_age_group"] = pd.cut(
        df["age"], bins=age_bins, labels=age_labels, include_lowest=True
    )
    age_metrics = [
        c for c in [
            "income", "analysis_total_expenses", "disposable_income",
            "desired_savings", "dependents"
        ] if c in df.columns
    ]
    age_summary = df.groupby(
        "analysis_age_group", observed=False, dropna=False
    )[age_metrics].agg(["count", "mean", "median"])
    display(age_summary)

    age_plot = df.groupby("analysis_age_group", observed=False)[
        "analysis_total_expenses"
    ].mean()
    ax = age_plot.plot(
        kind="bar", figsize=(9, 5),
        title="Average Complete-Record Expenses by Age Group"
    )
    ax.set_xlabel("Age group")
    ax.set_ylabel("Average expenses")
    plt.tight_layout()
    plt.savefig(CHARTS_DIR / "05_average_expenses_by_age_group.png", dpi=150, bbox_inches="tight")
    plt.show()

    outside_bins = int(df["age"].notna().sum() - df["analysis_age_group"].notna().sum())
    print("Non-missing ages outside defined bins:", outside_bins)

Interpretation: Compare group sizes as well as averages. Age groups are descriptive segments, not evidence that age causes spending differences.

## 9. Business Question 5 — Occupation Comparison

In [ ]:
if "occupation" not in df.columns:
    print("Occupation column unavailable; skipped.")
else:
    metrics = [
        c for c in [
            "income", "analysis_total_expenses", "disposable_income", "desired_savings"
        ] if c in df.columns
    ]
    occupation_summary = df.groupby("occupation", dropna=False)[metrics].agg(
        ["count", "mean", "median"]
    )
    display(occupation_summary)

    occupation_income = df.groupby("occupation", dropna=False)["income"].median().dropna().sort_values()
    if not occupation_income.empty:
        ax = occupation_income.plot(
            kind="barh", figsize=(9, 5), title="Median Income by Occupation"
        )
        ax.set_xlabel("Median income")
        plt.tight_layout()
        plt.savefig(CHARTS_DIR / "06_median_income_by_occupation.png", dpi=150, bbox_inches="tight")
        plt.show()

Interpretation: Median income is less sensitive to extreme values. Check group sizes before conclusions. Missing occupation values are retained as a group.

## 10. Business Question 6 — City Tier Comparison

In [ ]:
if "city_tier" not in df.columns:
    print("City tier column unavailable; skipped.")
else:
    metrics = [
        c for c in [
            "income", "analysis_total_expenses", "rent", "groceries",
            "transport", "disposable_income"
        ] if c in df.columns
    ]
    city_summary = df.groupby("city_tier", dropna=False)[metrics].agg(
        ["count", "mean", "median"]
    )
    display(city_summary)

    if "rent" in df.columns:
        rent_by_city = df.groupby("city_tier", dropna=False)["rent"].median().dropna().sort_values()
        if not rent_by_city.empty:
            ax = rent_by_city.plot(kind="bar", figsize=(8, 5), title="Median Rent by City Tier")
            ax.set_xlabel("City tier")
            ax.set_ylabel("Median rent")
            plt.tight_layout()
            plt.savefig(CHARTS_DIR / "07_median_rent_by_city_tier.png", dpi=150, bbox_inches="tight")
            plt.show()

Interpretation: Compare rent and expenses alongside income. Higher expenses do not automatically mean greater financial pressure.

## 11. Business Question 7 — Potential Savings

In [ ]:
if not potential_savings_columns:
    print("Potential-savings columns unavailable; skipped.")
else:
    potential_summary = pd.DataFrame({
        "reported_record_count": df[potential_savings_columns].count(),
        "total_potential_savings": df[potential_savings_columns].sum(min_count=1),
        "average_potential_savings": df[potential_savings_columns].mean(),
        "median_potential_savings": df[potential_savings_columns].median()
    }).sort_values("total_potential_savings", ascending=False)

    potential_summary.index = potential_summary.index.str.replace(
        "potential_savings_", "", regex=False
    )
    display(potential_summary)

    ax = potential_summary["total_potential_savings"].sort_values().plot(
        kind="barh", figsize=(10, 6), title="Potential Savings by Category"
    )
    ax.set_xlabel("Potential savings")
    plt.tight_layout()
    plt.savefig(CHARTS_DIR / "08_potential_savings_by_category.png", dpi=150, bbox_inches="tight")
    plt.show()

Interpretation: These are dataset-provided modeled opportunities, not guaranteed savings. Confirm the source calculation method before recommendations.

## 12. Business Question 8 — Disposable Income Reconciliation

In [ ]:
if "disposable_income" not in df.columns:
    print("Provided disposable_income unavailable; skipped.")
else:
    reconciliation = df[[
        "income", "analysis_total_expenses",
        "analysis_estimated_remaining_income", "disposable_income"
    ]].copy()
    reconciliation["difference_vs_provided_disposable"] = (
        reconciliation["analysis_estimated_remaining_income"] -
        reconciliation["disposable_income"]
    )
    display(reconciliation.describe().T)

    diff = reconciliation["difference_vs_provided_disposable"].dropna()
    if not diff.empty:
        fig, ax = plt.subplots(figsize=(9, 5))
        sns.histplot(diff, bins=30, kde=True, ax=ax)
        ax.axvline(0, linestyle="--", color="red")
        ax.set_title("Estimated Remaining Income Minus Provided Disposable Income")
        ax.set_xlabel("Difference")
        plt.tight_layout()
        plt.savefig(CHARTS_DIR / "09_disposable_income_reconciliation.png", dpi=150, bbox_inches="tight")
        plt.show()

Interpretation: The estimate is income minus all available expense categories and is calculated only for records with complete expense data. Source disposable income may use another formula. A difference is a reconciliation question, not automatically an error.

## 13. Key Findings — Automatically Generated Evidence

In [ ]:
findings = [{
    "business_question": "Dataset overview",
    "finding": f"The dataset contains {len(df):,} records and {df.shape[1]:,} columns, including analysis metrics."
}]

if df["income"].notna().any():
    findings.append({
        "business_question": "Income distribution",
        "finding": f"Median income is {df['income'].median():,.2f}; mean income is {df['income'].mean():,.2f}."
    })

if expense_columns:
    totals = df[expense_columns].sum(min_count=1).dropna()
    if not totals.empty:
        findings.append({
            "business_question": "Highest reported expense category",
            "finding": f"{totals.idxmax()} has the highest total reported spending ({totals.max():,.2f})."
        })

if "occupation" in df.columns:
    medians = df.groupby("occupation", dropna=False)["income"].median().dropna()
    if not medians.empty:
        findings.append({
            "business_question": "Occupation comparison",
            "finding": f"{medians.idxmax()} has the highest median income ({medians.max():,.2f})."
        })

if "city_tier" in df.columns and "rent" in df.columns:
    medians = df.groupby("city_tier", dropna=False)["rent"].median().dropna()
    if not medians.empty:
        findings.append({
            "business_question": "City-tier rent comparison",
            "finding": f"{medians.idxmax()} has the highest median rent ({medians.max():,.2f})."
        })

if potential_savings_columns:
    totals = df[potential_savings_columns].sum(min_count=1).dropna()
    if not totals.empty:
        top = totals.idxmax().replace("potential_savings_", "")
        findings.append({
            "business_question": "Largest modeled savings opportunity",
            "finding": f"{top} has the highest total potential savings ({totals.max():,.2f})."
        })

findings_df = pd.DataFrame(findings, columns=["business_question", "finding"])
display(findings_df)

## 14. Export EDA Results

In [ ]:
summary_rows = []
for column in [
    "income", "disposable_income", "desired_savings",
    "desired_savings_percentage", "analysis_total_expenses",
    "analysis_estimated_remaining_income", "analysis_expense_ratio_pct"
]:
    if column in df.columns:
        summary_rows.append({
            "metric": column,
            "count": int(df[column].count()),
            "mean": df[column].mean(),
            "median": df[column].median(),
            "min": df[column].min(),
            "max": df[column].max()
        })

eda_summary = pd.DataFrame(
    summary_rows, columns=["metric", "count", "mean", "median", "min", "max"]
)
eda_summary.to_csv(REPORTS_DIR / "eda_summary.csv", index=False)
findings_df.to_csv(REPORTS_DIR / "business_insights.csv", index=False)

print("EDA summary:", (REPORTS_DIR / "eda_summary.csv").resolve())
print("Business insights:", (REPORTS_DIR / "business_insights.csv").resolve())
print("Charts folder:", CHARTS_DIR.resolve())

## 15. Final Review and Portfolio Notes

- Run all three notebooks from top to bottom.
- Confirm the reports and chart files are generated.
- Review missing values and category counts.
- Expense-based total calculations require all available expense categories to be present.
- Verify the source business rules for disposable income and potential savings.
- Replace generic notes with findings supported by your actual dataset.
- Do not claim causation from descriptive statistics or correlations.

**Project workflow:** `raw_data.csv` → Exploration → Cleaning → `cleaned_data.csv` → EDA → Reports and visualizations.

**Scope:** Python, Pandas, NumPy, Matplotlib, Seaborn, data quality, statistical analysis, and business interpretation.